In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Data

In [2]:
df_train = pd.read_csv('data/train.csv')
df_test = pd.read_csv('data/test.csv')
df_original = pd.read_csv('data/Rainfall.csv')

In [3]:
df_train.head()

,id,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed,rainfall
0,0,1,1017.4,21.2,20.6,19.9,19.4,87.0,88.0,1.1,60.0,17.2,1
1,1,2,1019.5,16.2,16.9,15.8,15.4,95.0,91.0,0.0,50.0,21.9,1
2,2,3,1024.1,19.4,16.1,14.6,9.3,75.0,47.0,8.3,70.0,18.1,1
3,3,4,1013.4,18.1,17.8,16.9,16.8,95.0,95.0,0.0,60.0,35.6,1
4,4,5,1021.8,21.3,18.4,15.2,9.6,52.0,45.0,3.6,40.0,24.8,0


- we remove 'id' column
- 'rainfall' variable is our target

In [4]:
train = df_train.copy()
train.drop(['id'], axis=1, inplace=True)

In [5]:
train.head()

,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed,rainfall
0,1,1017.4,21.2,20.6,19.9,19.4,87.0,88.0,1.1,60.0,17.2,1
1,2,1019.5,16.2,16.9,15.8,15.4,95.0,91.0,0.0,50.0,21.9,1
2,3,1024.1,19.4,16.1,14.6,9.3,75.0,47.0,8.3,70.0,18.1,1
3,4,1013.4,18.1,17.8,16.9,16.8,95.0,95.0,0.0,60.0,35.6,1
4,5,1021.8,21.3,18.4,15.2,9.6,52.0,45.0,3.6,40.0,24.8,0


In [6]:
X_train = train.drop('rainfall', axis=1)
Y_train = train['rainfall']
target = 'rainfall'

In [7]:
df_original.head()

,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,rainfall,sunshine,winddirection,windspeed
0,1,1025.9,19.9,18.3,16.8,13.1,72,49,yes,9.3,80.0,26.3
1,2,1022.0,21.7,18.9,17.2,15.6,81,83,yes,0.6,50.0,15.3
2,3,1019.7,20.3,19.3,18.0,18.4,95,91,yes,0.0,40.0,14.2
3,4,1018.9,22.3,20.6,19.1,18.8,90,88,yes,1.0,50.0,16.9
4,5,1015.9,21.3,20.7,20.2,19.9,95,81,yes,0.0,40.0,13.7


In [8]:
df_original.columns = df_original.columns.str.strip()
original = df_original.copy()
original['rainfall'] = original['rainfall'].map({'no': 0, 'yes': 1})

In [9]:
X_original = original.drop('rainfall', axis=1)
Y_original = original['rainfall']

## Adding weather information from previous days

We add 3 types of new columns: weather features from the previous day, the average values of features from the previous two days, and the average values from the previous three days. 
Among the average values, ​​we exclude the variables: 'winddirection' and 'rainfall'.

In [10]:
train_shifted1 = train.shift(1)
train_shifted1.drop(['day'], axis=1, inplace=True)

train_prev1 = pd.concat([train, train_shifted1.add_prefix("prev1_")], axis=1)

train_shifted1.drop(['winddirection', 'rainfall'], axis=1, inplace=True)

train_shifted2 = train.shift(2)
train_shifted2.drop(['day', 'winddirection', 'rainfall'], axis=1, inplace=True)

train_shifted3 = train.shift(3)
train_shifted3.drop(['day', 'winddirection', 'rainfall'], axis=1, inplace=True)

In [11]:
train_avg2 = pd.concat([train_shifted1, train_shifted2]).groupby(level=0).mean()
train_avg2[(train_shifted1.isna()) | (train_shifted2.isna())] = np.nan
train_avg2 = train_avg2.add_prefix("avg2_")
train_avg2 = train_avg2.round(1)

train_avg3 = pd.concat([train_shifted1, train_shifted2, train_shifted3]).groupby(level=0).mean()
train_avg3[(train_shifted1.isna()) | (train_shifted2.isna()) | (train_shifted3.isna())] = np.nan
train_avg3 = train_avg3.add_prefix("avg3_")
train_avg3 = train_avg3.round(1)

In [12]:
train_prev = pd.concat([train_prev1, train_avg2, train_avg3], axis=1)
# removing the first three records that lack information from the previous three days
train_prev = train_prev.dropna().reset_index(drop=True)
train_prev.head()

,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,...,avg2_windspeed,avg3_pressure,avg3_maxtemp,avg3_temparature,avg3_mintemp,avg3_dewpoint,avg3_humidity,avg3_cloud,avg3_sunshine,avg3_windspeed
0,4,1013.4,18.1,17.8,16.9,16.8,95.0,95.0,0.0,60.0,...,20.0,1020.3,18.9,17.9,16.8,14.7,85.7,75.3,3.1,19.1
1,5,1021.8,21.3,18.4,15.2,9.6,52.0,45.0,3.6,40.0,...,26.8,1019.0,17.9,16.9,15.8,13.8,88.3,77.7,2.8,25.2
2,6,1022.7,20.6,18.6,16.5,12.5,79.0,81.0,0.0,20.0,...,30.2,1019.8,19.6,17.4,15.6,11.9,74.0,62.3,4.0,26.2
3,7,1022.8,19.5,18.4,15.3,11.3,56.0,46.0,7.6,20.0,...,20.2,1019.3,20.0,18.3,16.2,13.0,75.3,73.7,1.2,25.4
4,8,1019.7,15.8,13.6,12.7,11.8,96.0,100.0,0.0,50.0,...,22.0,1022.4,20.5,18.5,15.7,11.1,62.3,57.3,3.7,23.0


In [13]:
X_train_prev = train_prev.drop('rainfall', axis=1)
Y_train_prev = train_prev['rainfall']

## Checking importance of new columns

### Random Forest

In [14]:
from sklearn.ensemble import RandomForestClassifier

**Original data**

In [15]:
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, Y_train)

importances = clf.feature_importances_
features_importance_org = pd.Series(importances, index=X_train.columns).sort_values(ascending=False)

print(features_importance_org)

cloud            0.303654
sunshine         0.158596
humidity         0.096387
dewpoint         0.061703
windspeed        0.060770
day              0.058626
maxtemp          0.058191
pressure         0.055699
mintemp          0.054301
temparature      0.054286
winddirection    0.037788
dtype: float64


**Data with columns from the previous day and the average values from the previous two and three days**

In [16]:
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train_prev, Y_train_prev)

importances = clf.feature_importances_
features_importance_prev = pd.Series(importances, index=X_train_prev.columns).sort_values(ascending=False)

print(features_importance_prev)

cloud                  0.182568
sunshine               0.133763
humidity               0.090072
dewpoint               0.025505
windspeed              0.023294
temparature            0.022245
avg3_cloud             0.022167
maxtemp                0.020473
day                    0.020003
pressure               0.019773
avg3_sunshine          0.019112
prev1_windspeed        0.018698
prev1_sunshine         0.017745
avg3_windspeed         0.017627
avg2_humidity          0.017357
mintemp                0.017184
avg2_sunshine          0.016970
avg2_cloud             0.016583
avg3_humidity          0.016031
avg3_dewpoint          0.015979
avg2_mintemp           0.015580
prev1_mintemp          0.015318
avg3_pressure          0.015075
prev1_maxtemp          0.015012
avg2_windspeed         0.014915
avg2_dewpoint          0.014840
prev1_cloud            0.014791
prev1_pressure         0.014476
prev1_dewpoint         0.014452
winddirection          0.014017
prev1_temparature      0.013410
avg3_max